# Brain MRI Project

## 0) Usefull things, imports

In [43]:
#load the autoreload extension (magic command, only for IPython)
%load_ext autoreload 
#before each cell, re-execute any imported module whose file changed on disk
%autoreload 2

import numpy as np
import pandas as pd

# --- model selection / evaluation
from sklearn.model_selection import KFold, GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score

# --- scaling
from sklearn.preprocessing import StandardScaler, RobustScaler

# --- imputation
from sklearn.experimental import enable_iterative_imputer   # noqa: F401  (must precede the next line)
from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer

# --- outlier detection (sklearn: +1 inlier / -1 outlier)
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.covariance import EllipticEnvelope, MinCovDet
from sklearn.svm import OneClassSVM

# --- outlier detection (PyOD: 0 inlier / 1 outlier)  -- pip install pyod
from pyod.models.ecod import ECOD
from pyod.models.copod import COPOD

# --- feature selection
from sklearn.feature_selection import (
    VarianceThreshold, SelectKBest, SelectFromModel,
    f_regression, mutual_info_regression, RFE, RFECV,
)

# --- models
from sklearn.linear_model import Ridge, Lasso, ElasticNet, HuberRegressor
from sklearn.cross_decomposition import PLSRegression
from sklearn.base import BaseEstimator, RegressorMixin, clone

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1) Loading data


In [44]:

X_train_df = pd.read_csv('./data/X_train.csv', skiprows=1, header=None)
y_train_df = pd.read_csv('./data/y_train.csv', skiprows=1, header=None)
X_test_df = pd.read_csv('./data/X_test.csv', skiprows=1, header=None)

# drop the id column
X_train = X_train_df.values[:, 1:]
y_train = y_train_df.values[:, 1:]
X_test = X_test_df.values[:, 1:]
print(X_train.shape, y_train.shape, X_test.shape)

print(np.isnan(X_test).sum().sum())

#remember to normalize before using methods

(1212, 832) (1212, 1) (776, 832)
40634


# 2) Preprocessing

## A) Dropping low data features (filter)

In [45]:
#inspection
pd.Series(np.isnan(X_train).mean(axis=1)).sort_values(ascending=False)
pd.Series(np.isnan(X_train).mean(axis=0)).sort_values(ascending=False)

#no clear break in the distribution of missing values (seems to be a binomial distribution), so we will not remove any rows or columns

79     0.099835
381    0.098185
734    0.097360
313    0.097360
463    0.096535
         ...   
547    0.056931
728    0.056106
425    0.056106
151    0.054455
703    0.053630
Length: 832, dtype: float64

## B) Remove outliers

The mean and variance/covariance are contaminated by the outliers => must remove outliers before converting the Nans, otherwise we fit information that's not from the data set.

But to find the outliers we must first eliminate Nans temporarly with a method robust to the outliers (eg median).

!!! test set has no outliers => check if letting outliers or not in validation test affects score (could lead to higher lambdas in ridge), if yes then remove outliers from it too, but careful of degenerate incentive (method of removing outlier tuned from validation => validation set changes with it => exam gets shorter and easier) => fixed method for validation for identical exam for each candidate



In [46]:
#we will iterate over different methods to remove outliers and keep the one that scores the best
#note : we ll start with just a few not too heavy methods and then add if needed. Methods are stored in order of their complexity, 'l' is the nb of methods used
l_train = 3
l_val = 2

out_methods_train = [
    IsolationForest(contamination=0.05, random_state=42),   # 1. best-matched to high d, O(n log n)
    ECOD(contamination=0.05),                               # 2. parameter-free, fast        [PyOD]
    COPOD(contamination=0.05),                              # 3. parameter-free, sees dependence [PyOD]
    "zscore",                                               # 4. univariate baseline — to do
    LocalOutlierFactor(n_neighbors=20, novelty=True,
                       contamination=0.05),                 # 5. only if intrinsic dim is low
    EllipticEnvelope(contamination=0.05, random_state=42),  # 6. Mahalanobis+MCD — needs d reduced first
    OneClassSVM(nu=0.05, kernel="rbf", gamma="scale"),      # 7. expensive, no contamination param
    "robust_residuals",                                     # 8. Huber fit + large residuals — to do
]

out_methods_val = [
    None,                                                    # dirty validation
    IsolationForest(contamination=0.05, random_state=0),     # frozen referee — chosen once, never tuned
]
out_methods_val_names = ["None on validation: ", "IsolationForest on validation: "]

#the following was my original architecture. I changed to a wrapper class to use it inside gridsearchcv
#First eliminate the nans with a method robust to outliers (only to find them) : impute median
def nan_to_median(X_train):
    X_median = np.nanmedian(X_train, axis=0, keepdims=True)
    X_mask = np.isnan(X_train)
    X_train = np.where(X_mask, X_median, X_train)

    return (X_train, X_mask), X_median

#below are old helpers i built
"""
#outliers of val removed independently of the train set, to avoid data leakage
def remove_outliers_val(X_val, val_maskmethod_val):
    if method_val is None:
        return X_val
    #remove outliers using the method
    return X_val

#outliers of train fune-tuned to the val set
def remove_outliers_train(X_train, y_train, train_mask, method_train):
    #remove outliers using the method
    return X_train

#after removing outliers, we put the nans back for conditionnal imputation
def replace_nans(X, mask):
    return np.where(mask, np.nan, X)

def remove_outliers(method_train, method_val, X_train, y_train, X_val):
    (X_train, train_mask), (X_val, val_mask) = nan_to_median(X_train, X_val)

    X_val, val_mask = remove_outliers_val(X_val, val_mask, method_val)
    X_train, y_train, train_mask = remove_outliers_train(X_train, y_train, train_mask,method_train)

    X_train, X_val = replace_nans(X_train, train_mask), replace_nans(X_val, val_mask)

    return X_train, X_val
"""

#wrapper class to use this outlier removal methods inside gridsearchcv
class OutlierRemover(BaseEstimator, RegressorMixin):
    def __init__(self, inner=None, detector=None, referee=None, contamination=0.05):
        self.inner = inner
        self.detector = detector
        self.referee = referee
        self.contamination = contamination
        self.med_=None

    def fit(self, X, y):
        #impute nans with median (robust to outliers)
        (X_clean, X_nan_mask), med = nan_to_median(X) 
        self.med_ = med

        #outlier removal of validation set (fit on training set to avoid leakage, method frozen to keep the same exam)
        if self.referee is not None:
            self.referee_ = clone(self.referee).fit(X_clean, y) 
        else :
            self.referee_ = None

        #outlier removal of training set
        detector = clone(self.detector)

        # map self.contamination to the appropriate parameter name for the detector
        valid_params = detector.get_params()
        if 'contamination' in detector.get_params():
            detector.set_params(contamination=self.contamination)
        elif 'nu' in valid_params:
            detector.set_params(nu=self.contamination)

        detector.fit(X_clean, y) #remove outliers
        keep = keep_mask(detector, X_clean) #value of outlier depends on the detector
        self.inner_ = clone(self.inner).fit(X[keep], y[keep])

        return self

    def predict(self, X):
        return self.inner_.predict(X) 


def keep_mask(detector, X):     
    # PyOD estimators belong to the 'pyod' module and expose 'decision_scores_'
    is_pyod = type(detector).__module__.startswith('pyod') or hasattr(detector, 'decision_scores_')
    predictions = detector.predict(X)
        
    keep = np.ones(X.shape[0], dtype=bool)  
    if is_pyod:
        # PyOD convention: 0 = inlier, 1 = outlier
        keep = (predictions == 0)
    else:
        # Scikit-learn convention: 1 = inlier, -1 = outlier
        keep = (predictions == 1)
        
    #Check we aren't deleting most of the dataset due to a configuration error
    if keep.mean() < 0.5:   
        raise ValueError("More than 50% of the data was flagged as outliers, check the detector configuration.")
    return keep


#outlier removal of the validation set directly in the scorer here !
def frozen_referee_as_scorer():

    def frozen_referee_r2(estimator, X_val, y_val, out_method_val=None): 
        if estimator.referee_ is None:
            keep = np.ones(len(y_val), dtype=bool)
        else:
            X   = np.where(np.isnan(X_val), estimator.med_, X_val)   # training medians
            keep = keep_mask(estimator.referee_, X)
        return r2_score(y_val[keep], estimator.predict(X_val[keep]))
    
    return frozen_referee_r2


## C) Impute Nans

1) Constant imputation : eg replacing by the mean : reduces the variance by (1-p)sj^2 with p the fraction replaced, and covariance. Prediction pulled towards the mean (biais attenuation). Might give impossible values (eg feature surface and volume physically impossible together)
2) Conditionnal imputation : predict missing values from other features (keep correlation structure) -> Better
eg : KNNImputer (avg k similar subjects) ; IterativeImputer (regress each feature on the others)



In [47]:
l_impute = 2

imputation_methods = [
    SimpleImputer(strategy="median"),            # 1. baseline 
    KNNImputer(n_neighbors=3), KNNImputer(n_neighbors=5), KNNImputer(n_neighbors=10),                  # 2. conditional, cheap
    IterativeImputer(max_iter=5, random_state=42),  # 3. conditional, expensive at d=832
]

def impute_naans(method, X_train, X_val, X_test=None):
    #impute the nans using the method
    return X_train, X_val, X_test

## D) Feature selection

Why SVD fails:

Total intracranial volume varies enormously across subjects — and is essentially age-independent (skull size is set in childhood).
Ventricular volume varies much less — and is one of the strongest age markers (ventricles enlarge with age).

PC1 is then dominated by head size. Truncate to 10 components might even delete the ventricular volume component.
Irrelevant features were added => If they have substantial variance, SVD ranks them first. (+any outlier left can have high variance)

In [48]:
l_feature_indep_y = 1
l_feature_dep_y = 3

feature_methods_indep_y = [
    VarianceThreshold(threshold=0.0),   # drops constant columns
    "correlation_pruning",              # to do
]

feature_methods_dep_y = [
    SelectKBest(f_regression, k=50), SelectKBest(f_regression, k=100),
    SelectKBest(f_regression, k=200), VarianceThreshold(0.0),
    SelectKBest(mutual_info_regression, k=100),
    SelectFromModel(ElasticNet(alpha=0.1, l1_ratio=0.5)),   # embedded — the best-matched one
    RFE(Ridge(alpha=1.0), n_features_to_select=100, step=0.1),  # expensive, add later
]
#PLS later? (covariance with y truncates irrelevant features)

def feature_selection(X_train, y_train, X_val, X_test, method_indep_y, method_dep_y):
    #apply the methods to select features
    return X_train, X_val, X_test

## E) Model

We use ridge regression because ordinary linear regression highly underestimates the variance on the training set (s^2(1-d/n) vs s^2(1+d/n) for an indep test set) and multicolinearity (like for n>~d) makes the det~0 => analytical solution can diverge (the solution is a whole subspace)

In [49]:
l_models = 2

#models are iterated on, i put the different hyperparameters here for now, not ideal but avoids needing to unify model__para for different models with different parameters in gridsearchcv

models = [
    Ridge(alpha=0.1), Ridge(alpha=1.0), Ridge(alpha=10), Ridge(alpha=50),
    ElasticNet(alpha=1, l1_ratio=0.5), ElasticNet(alpha=0.1, l1_ratio=1),
    HuberRegressor(epsilon=1.35, alpha=1e-4),   # the robust-loss arm of the 2×2
    PLSRegression(n_components=20),             # only after outlier removal
]

#("model", RidgeCV(alphas=np.logspace(-3, 4, 15))) can be used to sweep different alphas for ridge fast

## F) Pipeline settings, parameters to tune

In [50]:
#pipeline (each step can be replaced by a different method, to be tuned with grid search)


pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")), #only one now, will be replaced by grid search from the grid (clone)
    ("scale",  StandardScaler()),
    ("feature", SelectKBest(f_regression, k=100)),
    ("model",  Ridge(alpha=1.0)),
])

#parameter grid for grid search (found using pipe.get_params().keys())
#referee outisde the grid (frozen) to have the same exam
grid = {
    #depth 1:
    "detector":      out_methods_train[:l_train],
    "contamination":   [0.05, 0.1, 0.2],
    #depth 2 (para of the inner pipeline):
    "inner__impute":        imputation_methods[:l_impute],
    "inner__feature":       feature_methods_indep_y[:l_feature_indep_y]+feature_methods_dep_y[:l_feature_dep_y],
    "inner__model":         models[:l_models],
    #depth 3:
    #"inner__model__alpha":  [0.1,1.0, 10.0, 50.0],
    #"inner__feature__k":      [50, 100, 200, 400],
    #"inner__model__l1_ratio": [0.1, 0.5, 0.9],
    #"inner__impute__n_neighbors": [3, 5, 10]
}

## G) Train the model and predict

In [51]:
results = {}
kf = KFold(5, shuffle=True, random_state=42)

for name, referee in zip(out_methods_val_names[:l_val], out_methods_val[:l_val]):
    search = GridSearchCV(
        OutlierRemover(inner=pipe, detector=IsolationForest(), referee=referee),
        grid, scoring=frozen_referee_as_scorer(), cv=kf,
        n_jobs=-1, verbose=2, error_score=np.nan, return_train_score=True,
    )
    search.fit(X_train, y_train.ravel())

    results[name] = search
    print(name, search.best_params_)

Fitting 5 folds for each of 144 candidates, totalling 720 fits
[CV] END contamination=0.05, detector=IsolationForest(contamination=0.05, random_state=42), inner__feature=VarianceThreshold(), inner__impute=SimpleImputer(strategy='median'), inner__model=Ridge(alpha=0.1); total time=   0.8s
[CV] END contamination=0.05, detector=IsolationForest(contamination=0.05, random_state=42), inner__feature=VarianceThreshold(), inner__impute=SimpleImputer(strategy='median'), inner__model=Ridge(); total time=   0.8s
[CV] END contamination=0.05, detector=IsolationForest(contamination=0.05, random_state=42), inner__feature=VarianceThreshold(), inner__impute=SimpleImputer(strategy='median'), inner__model=Ridge(); total time=   0.8s
[CV] END contamination=0.05, detector=IsolationForest(contamination=0.05, random_state=42), inner__feature=VarianceThreshold(), inner__impute=SimpleImputer(strategy='median'), inner__model=Ridge(alpha=0.1); total time=   0.8s
[CV] END contamination=0.05, detector=IsolationFore

/opt/anaconda3/envs/amlproject1/lib/python3.12/site-packages/pyod/models/base.py:555: UserWarning: y should not be presented in unsupervised learning.
  warnings.warn(


[CV] END contamination=0.05, detector=IsolationForest(contamination=0.05, random_state=42), inner__feature=SelectKBest(k=200, score_func=<function f_regression at 0x13117a200>), inner__impute=KNNImputer(n_neighbors=3), inner__model=Ridge(alpha=0.1); total time=   4.7s
[CV] END contamination=0.05, detector=IsolationForest(contamination=0.05, random_state=42), inner__feature=SelectKBest(k=200, score_func=<function f_regression at 0x11ef36200>), inner__impute=KNNImputer(n_neighbors=3), inner__model=Ridge(alpha=0.1); total time=   4.7s
[CV] END contamination=0.05, detector=IsolationForest(contamination=0.05, random_state=42), inner__feature=SelectKBest(k=200, score_func=<function f_regression at 0x1184fa200>), inner__impute=KNNImputer(n_neighbors=3), inner__model=Ridge(alpha=0.1); total time=   4.6s
[CV] END contamination=0.05, detector=IsolationForest(contamination=0.05, random_state=42), inner__feature=SelectKBest(k=200, score_func=<function f_regression at 0x12ada6200>), inner__impute=K

/opt/anaconda3/envs/amlproject1/lib/python3.12/site-packages/pyod/models/base.py:555: UserWarning: y should not be presented in unsupervised learning.
  warnings.warn(
/opt/anaconda3/envs/amlproject1/lib/python3.12/site-packages/pyod/models/base.py:555: UserWarning: y should not be presented in unsupervised learning.
  warnings.warn(
/opt/anaconda3/envs/amlproject1/lib/python3.12/site-packages/pyod/models/base.py:555: UserWarning: y should not be presented in unsupervised learning.
  warnings.warn(
/opt/anaconda3/envs/amlproject1/lib/python3.12/site-packages/pyod/models/base.py:555: UserWarning: y should not be presented in unsupervised learning.
  warnings.warn(
/opt/anaconda3/envs/amlproject1/lib/python3.12/site-packages/pyod/models/base.py:555: UserWarning: y should not be presented in unsupervised learning.
  warnings.warn(


[CV] END contamination=0.05, detector=IsolationForest(contamination=0.05, random_state=42), inner__feature=SelectKBest(k=200, score_func=<function f_regression at 0x116452200>), inner__impute=KNNImputer(n_neighbors=3), inner__model=Ridge(alpha=0.1); total time=   4.9s
[CV] END contamination=0.05, detector=IsolationForest(contamination=0.05, random_state=42), inner__feature=SelectKBest(k=200, score_func=<function f_regression at 0x11e2de200>), inner__impute=KNNImputer(n_neighbors=3), inner__model=Ridge(); total time=   4.7s
[CV] END contamination=0.05, detector=IsolationForest(contamination=0.05, random_state=42), inner__feature=SelectKBest(k=200, score_func=<function f_regression at 0x120956200>), inner__impute=KNNImputer(n_neighbors=3), inner__model=Ridge(); total time=   4.8s
[CV] END contamination=0.05, detector=IsolationForest(contamination=0.05, random_state=42), inner__feature=SelectKBest(k=200, score_func=<function f_regression at 0x121572200>), inner__impute=KNNImputer(n_neighb

/opt/anaconda3/envs/amlproject1/lib/python3.12/site-packages/pyod/models/base.py:555: UserWarning: y should not be presented in unsupervised learning.
  warnings.warn(
/opt/anaconda3/envs/amlproject1/lib/python3.12/site-packages/pyod/models/base.py:555: UserWarning: y should not be presented in unsupervised learning.
  warnings.warn(
/opt/anaconda3/envs/amlproject1/lib/python3.12/site-packages/pyod/models/base.py:555: UserWarning: y should not be presented in unsupervised learning.
  warnings.warn(
/opt/anaconda3/envs/amlproject1/lib/python3.12/site-packages/pyod/models/base.py:555: UserWarning: y should not be presented in unsupervised learning.
  warnings.warn(


... (309 outputs left out)

/opt/anaconda3/envs/amlproject1/lib/python3.12/site-packages/pyod/models/base.py:555: UserWarning: y should not be presented in unsupervised learning.
  warnings.warn(
/opt/anaconda3/envs/amlproject1/lib/python3.12/site-packages/pyod/models/base.py:555: UserWarning: y should not be presented in unsupervised learning.
  warnings.warn(
/opt/anaconda3/envs/amlproject1/lib/python3.12/site-packages/pyod/models/base.py:555: UserWarning: y should not be presented in unsupervised learning.
  warnings.warn(
/opt/anaconda3/envs/amlproject1/lib/python3.12/site-packages/pyod/models/base.py:555: UserWarning: y should not be presented in unsupervised learning.
  warnings.warn(


[CV] END contamination=0.2, detector=ECOD(contamination=0.05, n_jobs=1), inner__feature=SelectKBest(k=50, score_func=<function f_regression at 0x13117a200>), inner__impute=SimpleImputer(strategy='median'), inner__model=Ridge(); total time=   1.7s


/opt/anaconda3/envs/amlproject1/lib/python3.12/site-packages/pyod/models/base.py:555: UserWarning: y should not be presented in unsupervised learning.
  warnings.warn(
/opt/anaconda3/envs/amlproject1/lib/python3.12/site-packages/pyod/models/base.py:555: UserWarning: y should not be presented in unsupervised learning.
  warnings.warn(
/opt/anaconda3/envs/amlproject1/lib/python3.12/site-packages/pyod/models/base.py:555: UserWarning: y should not be presented in unsupervised learning.
  warnings.warn(
/opt/anaconda3/envs/amlproject1/lib/python3.12/site-packages/pyod/models/base.py:555: UserWarning: y should not be presented in unsupervised learning.
  warnings.warn(
/opt/anaconda3/envs/amlproject1/lib/python3.12/site-packages/pyod/models/base.py:555: UserWarning: y should not be presented in unsupervised learning.
  warnings.warn(
/opt/anaconda3/envs/amlproject1/lib/python3.12/site-packages/pyod/models/base.py:555: UserWarning: y should not be presented in unsupervised learning.
  warning

KeyboardInterrupt: 

## H) Submission

In [ ]:
best = results["IsolationForest on validation: "].best_estimator_      #need to check the best one first 
y_pred = best.predict(X_test)               

submission = pd.DataFrame({"id": np.arange(len(y_pred)), "y": y_pred.ravel()})
submission.to_csv("submissions/sub_01.csv", index=False)
print(submission.shape, submission.head())